# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook demonstrates how to load, inspect, and analyze the FAIR^2 mass spectrometry dataset of extracellular vesicles using the `mlcroissant` library, referencing all schema entities by their `@id` identifiers.

### Dataset Source
The Croissant schema for this dataset is located at:

`https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json`

In [ ]:
# Ensure `mlcroissant` is installed
!pip install mlcroissant --quiet

## 1. Data Loading
Load the dataset schema and metadata from the Croissant URL.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata and schema
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata

print("Dataset Title:", metadata.name)
print("Description:", metadata.description)
print("Identifier:", getattr(metadata, 'identifier', None))
print("Published Date:", getattr(metadata, 'datePublished', None))
print("Fields included:", getattr(metadata, 'keywords', []))

## 2. Data Overview
List the available record sets (`cr:RecordSet`) and their fields, referencing all by their `@id` as prescribed in the Croissant schema.

In [ ]:
# List all record sets by @id
record_sets = getattr(metadata, 'recordSet', [])
if isinstance(record_sets, dict):  # Just one record set
    record_sets = [record_sets]

print(f"Found {len(record_sets)} record set(s):")
for rec in record_sets:
    print(f"  RecordSet @id: {rec['@id'] if isinstance(rec, dict) else rec}")

# For demonstration, print schema overview for each record set
for rec in record_sets:
    rec_id = rec['@id'] if isinstance(rec, dict) else rec
    rec_obj = dataset.metadata.find_by_id(rec_id)
    print(f"\nRecordSet: {rec_id}")
    if hasattr(rec_obj, 'field'):
        fields = rec_obj.field
        if isinstance(fields, dict) or isinstance(fields, str):
            fields = [fields]
        print("  Field @ids:")
        for field in fields:
            field_id = field['@id'] if isinstance(field, dict) else field
            print(f"    {field_id}")
    else:
        print("  No fields found.")

## 3. Data Extraction
Load the records from each record set into a pandas DataFrame. Refer to each record set and field using their `@id`s.

In [ ]:
# Collect all record set @ids
record_set_ids = [r['@id'] if isinstance(r, dict) else r for r in record_sets]
dataframes = dict()

# Extract records from each record set
for rec_id in record_set_ids:
    records = list(dataset.records(record_set=rec_id))
    df = pd.DataFrame(records)
    dataframes[rec_id] = df
    print(f"RecordSet {rec_id}")
    print(f"Columns: {df.columns.tolist()}")
    display(df.head())
    print("-"*60)

## 4. Exploratory Data Analysis (EDA)
Explore and process data, using numeric and categorical field `@id`s. This includes, for example, filtering on a numeric field and then normalizing and grouping.

*Note: Please modify field `@id`s in the code below to fit the actual field names listed above for your use case.*

In [ ]:
# Example: select the main record set and relevant fields by their @id
# For illustration, we guess a typical numeric field and a groupable field (edit as required):
main_rs_id = record_set_ids[0] if record_set_ids else None
df = dataframes[main_rs_id]

# You will need to adapt these to the true @id or DataFrame column names
numeric_field_id = 'coverage_percent'  # Example field @id (replace with real)
group_field_id = 'description'         # Example field @id (replace with real)

# Confirm the fields exist in dataframe
columns = df.columns.tolist()
print("Available columns:", columns)

if numeric_field_id in columns:
    threshold = 10
    filtered_df = df[df[numeric_field_id].astype(float) > threshold].copy()
    print(f"Filtered records from '{main_rs_id}' where {numeric_field_id} > {threshold}:")
    display(filtered_df.head())
    # Normalize numeric field
    filtered_df[f"{numeric_field_id}_normalized"] = (
        filtered_df[numeric_field_id].astype(float) - filtered_df[numeric_field_id].astype(float).mean()
    ) / filtered_df[numeric_field_id].astype(float).std()
    print(f"\nNormalized {numeric_field_id}:\n")
    display(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())
    # Group by a field if exists
    if group_field_id in filtered_df.columns:
        grouped_df = filtered_df.groupby(group_field_id).mean(numeric_only=True)
        print(f"\nGrouped by '{group_field_id}':")
        display(grouped_df.head())
else:
    print(f"Column '{numeric_field_id}' not found. Please update numeric_field_id to an existing field from: {columns}")

## 5. Visualization
Visualize numerical field distributions and group relationships. Adjust field `@id`s as needed.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if main_rs_id and numeric_field_id in df.columns:
    plt.figure(figsize=(8,4))
    sns.histplot(df[numeric_field_id].astype(float), bins=20, kde=True)
    plt.title(f"Distribution of {numeric_field_id} in RecordSet {main_rs_id}")
    plt.xlabel(numeric_field_id)
    plt.show()
    
    # If a groupable field exists, make boxplot
    if group_field_id in df.columns:
        plt.figure(figsize=(10,6))
        sns.boxplot(x=group_field_id, y=numeric_field_id, data=df)
        plt.title(f"{numeric_field_id} by {group_field_id}")
        plt.xticks(rotation=90)
        plt.show()
else:
    print(f"Field '{numeric_field_id}' not available for visualization. Please choose an available numeric field.")

## 6. Conclusion
This notebook illustrated how to access and process a FAIR mass spectrometry dataset via `mlcroissant`. We loaded the schema using its Croissant URL, extracted and explored the available record sets and fields by their `@id`, performed basic data exploration, and illustrated how to filter, normalize, and visualize the data.

**Next steps:** For in-depth analysis, consult the dataset's schema for detailed record set and field `@id`s and extend the EDA and visualization accordingly.